In [9]:
import pandas as pd 
import geopandas as gpd

CRS_DEFAULT = "EPSG: 4326"
CRS_COUNTRY_METRIC = "EPSG: 3347"

cities = pd.read_csv("../data/raw/worldcities.csv")

In [2]:
#select only cities of Canada
cities_country = cities[cities["country"] == "Canada"]

cities_country.to_csv("../data/processed/cities_country.csv", index=False)

In [3]:
#make it into geodataframe
cities_country_gdf = gpd.GeoDataFrame(
    cities_country, 
    geometry=gpd.points_from_xy(
        cities_country["lng"], 
        cities_country["lat"]),
    crs = CRS_DEFAULT)

In [4]:
#To make a surrounding area around cities we have to put the coordinates into meters
#EPSG Code needs to be adapted for the specific study region!
cities_country_gdf = cities_country_gdf.to_crs(CRS_COUNTRY_METRIC)

In [5]:
#primary cities and all admin cities have to be selected. 
primary_cities = cities_country_gdf[
    cities_country_gdf["capital"].isin(["admin","primary"])]

#Select all the cities but not those which are already in the primary_cities
other_cities = cities_country_gdf[
    ~cities_country_gdf.index.isin(primary_cities.index)]

#Next we have to sort the values so that the biggest city comes first
other_cities = other_cities.sort_values(
    "population",
    ascending=False
).reset_index(drop=True)

In [6]:
#make sort of a 20km buffer around the cities.
selected_cities = primary_cities.copy()

for index, city in other_cities.iterrows():
    distances = selected_cities.distance(city.geometry)
    min_distance = distances.min()
    
    if min_distance >= 150000:
        selected_cities = pd.concat(
            [selected_cities, city.to_frame().T],
            ignore_index=True)
        
    if len(selected_cities) == 41:
        break

In [7]:
selected_cities.to_csv("../data/processed/selected_cities_canada.csv", index=False)